# Colab POC

Verify VS Code ↔ Colab runtime + Google Drive access.

**Target file:** `1EVgLvJjnhqKbZLaEKUHCsZoNkpgpPJOJ`

## 1. Runtime Check

In [ ]:
import shutil, subprocess, sys

if shutil.which('nvidia-smi'):
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
    print('GPU:', r.stdout.strip() if r.returncode == 0 else 'none (CPU runtime)')
else:
    print('GPU: none (CPU runtime) — set Runtime > Change runtime type > GPU')
print('Python:', sys.version)

In [ ]:
!pip install -q gdown==5.2.0

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/UIT2026-DoAnCuoiKi'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root ready:', DRIVE_ROOT)

## 3. Download Shared File

In [ ]:
import gdown, pathlib

FILE_ID = '1EVgLvJjnhqKbZLaEKUHCsZoNkpgpPJOJ'
OUT_DIR = '/content/poc_download'
os.makedirs(OUT_DIR, exist_ok=True)

output_path = gdown.download(id=FILE_ID, output=OUT_DIR + '/', quiet=False, fuzzy=True)
print('Downloaded to:', output_path)

## 4. Inspect File

In [ ]:
p = pathlib.Path(output_path)
print(f'Name  : {p.name}')
print(f'Size  : {p.stat().st_size / 1024:.1f} KB')
print(f'Suffix: {p.suffix}')

if p.suffix == '.zip':
    import zipfile
    with zipfile.ZipFile(p) as z:
        print('\nContents:', z.namelist()[:20])
elif p.suffix in ('.tar', '.gz', '.tgz'):
    import tarfile
    with tarfile.open(p) as t:
        print('\nContents:', t.getnames()[:20])

## 6. Train POC — CSV Regression

Verify model training runs on the server: build a CSV, load with pandas, fit `LinearRegression`, report metrics.

In [ ]:
import numpy as np, pandas as pd

# Synthetic dataset: y = 3*x1 - 2*x2 + 5 + noise
rng = np.random.default_rng(42)
n = 500
x1 = rng.uniform(0, 10, n)
x2 = rng.uniform(0, 10, n)
y = 3 * x1 - 2 * x2 + 5 + rng.normal(0, 1.0, n)

df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})
CSV_PATH = '/content/poc_regression.csv'
df.to_csv(CSV_PATH, index=False)
print('CSV written:', CSV_PATH, df.shape)
df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(df['x1'], df['y'], s=8, alpha=0.5)
ax[0].set(xlabel='x1', ylabel='y', title='x1 vs y')
ax[1].scatter(df['x2'], df['y'], s=8, alpha=0.5, color='tab:orange')
ax[1].set(xlabel='x2', ylabel='y', title='x2 vs y')
ax[2].hist(df['y'], bins=30, color='tab:green', edgecolor='k')
ax[2].set(xlabel='y', ylabel='count', title='y distribution')
plt.tight_layout()
plt.show()

print(df.describe())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

data = pd.read_csv(CSV_PATH)
X = data[['x1', 'x2']].values
y = data['y'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

print('coef :', model.coef_, '(true: [3, -2])')
print('bias :', model.intercept_, '(true: 5)')
print('R2   :', r2_score(y_test, pred))
print('RMSE :', mean_squared_error(y_test, pred) ** 0.5)